In [1]:
import pandas as pd
from haversine import haversine, Unit

# Load the specific sheets from the Excel file
ATMInfo = pd.read_excel('Dataset-swapnilcopy.xlsx', sheet_name='Sheet8')
PdisInfo = pd.read_excel('Dataset-swapnilcopy.xlsx', sheet_name='Sheet1')

# Selecting necessary columns from the data
ATMInfo = ATMInfo[['address', 'bank', 'coord_x', 'coord_y']]
PdisInfo = PdisInfo[['police station name', 'x', 'y']]

# Function to find nearest police station for each ATM
def find_nearest_police_station(atm_coord, police_stations):
    min_distance = float('inf')  # Initialize with a large number
    nearest_police_station = None
    nearest_coords = None

    for _, police_station in police_stations.iterrows():
        police_coord = (police_station['x'], police_station['y'])
        # Calculate distance in kilometers and convert to meters
        distance = haversine(atm_coord, police_coord, unit=Unit.KILOMETERS) * 1000

        # Check if this distance is smaller than the current minimum distance
        if distance < min_distance:
            min_distance = distance
            nearest_police_station = police_station['police station name']
            nearest_coords = police_coord

    return nearest_police_station, nearest_coords, min_distance

# Create a new DataFrame to store the results
results = []

for _, atm in ATMInfo.iterrows():
    atm_coord = (atm['coord_x'], atm['coord_y'])
    
    # Call the function to get nearest police station information
    nearest_police_station, nearest_coords, distance_meters = find_nearest_police_station(atm_coord, PdisInfo)
    
    # Append the result in the same row
    results.append({
        'ATM Address': atm['address'],
        'Bank': atm['bank'],
        'ATM Coordinate (x, y)': atm_coord,
        'Nearest Police Station': nearest_police_station,
        'Police Station Coordinate (x, y)': nearest_coords,
        'Distance (meters)': distance_meters
    })

# Convert results into a DataFrame
nearest_police_df = pd.DataFrame(results)

# Set display format for float values to avoid scientific notation
pd.options.display.float_format = '{:,.2f}'.format

# Display the results
nearest_police_df.to_excel('nearest_police.xlsx')


In [2]:
import pandas as pd
data = pd.read_excel('nearest_police.xlsx')
data[['ATM Address',	'Bank',	'ATM Coordinate (x, y)',	'Nearest Police Station',	'Police Station Coordinate (x, y)',	'Distance (meters)']]

,ATM Address,Bank,"ATM Coordinate (x, y)",Nearest Police Station,"Police Station Coordinate (x, y)",Distance (meters)
0,Alameda das Linhas de Torres,Banco BPI,"(38.7728103, -9.1614562)",PSP - 41st Squadron (Lumiar),"(38.7763818, -9.1554179)",657.06
1,"Av. Dom Dinis 68, 2675-328",Millennium bcp,"(38.7872119, -9.180754)",PSP - 71st Precinct (Odivelas),"(38.78953, -9.1785879)",318.89
2,Praça Duque de Saldanha 31 - B,"Balcão Santander Saldanha, Lisboa","(38.7339979, -9.145642)",PSP - 21th Squadron (Campolide),"(38.7328938, -9.1592286)","1,184.87"
3,R. Barata Salgueiro 30 5,Citibank International Plc-branch in Portugal,"(38.7213309, -9.1487909)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",615.33
4,R. Castilho 26 Piso 8,Banco Best - Centro de Investimento de Lisboa,"(38.7216269, -9.1502042)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",501.62
...,...,...,...,...,...,...
106,R. dos Soeiros 339 D,Millennium bcp,"(38.7560318, -9.177233)",PSP - Posto Policial do Centro Comercial Colombo,"(38.7557185, -9.1884517)",973.42
107,Av. Eng. Duarte Pacheco 21B,Balcão Santander Work Café Amoreiras,"(38.724257, -9.1591484)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",508.45
108,Praça de Londres 3-B,novobanco Praça de Londres,"(38.7417434, -9.137523)",PSP - 12ª Esquadra (Olaias),"(38.7393457, -9.1259649)","1,037.29"
109,Alameda Manuel Ricardo Espírito Santo 2,EuroBic ABANCA - Benfica,"(38.7476, -9.18914)",PSP - 3rd Division of Lisbon,"(38.7457904, -9.1991176)",888.38


In [3]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

# Load the dataset
data = pd.read_excel('nearest_police.xlsx')

# Function to convert distance to meters
def convert_to_meters(distance_text):
    if 'km' in distance_text:
        # Convert kilometers to meters (1 km = 1000 meters)
        return float(distance_text.replace('km', '').strip()) * 1000
    elif 'm' in distance_text:
        # Distance is already in meters
        return float(distance_text.replace('m', '').strip())
    return float('inf')  # Return a large number in case something unexpected happens

# Function to extract the distance from Google Maps
def get_actual_distance(driver, atm_coords, police_coords):
    try:
        print(f"Fetching distance for ATM: {atm_coords} to Police: {police_coords}...")

        # Go to the Google Maps directions page
        driver.get("https://www.google.com/maps/dir///@15.3849045,73.8753416,15z?entry=ttu&g_ep=EgoyMDI0MDkwNC4wIKXMDSoASAFQAw%3D%3D")

        # Wait for the page to load
        time.sleep(3)
        
        # Find the "From" input field by XPath and enter the ATM coordinates
        from_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="sb_ifc50"]/input'))
        )
        from_input.clear()
        from_input.send_keys(atm_coords)
        from_input.send_keys("\n")  # Simulate pressing "Enter"
        
        # Wait for a second
        time.sleep(2)
        
        # Find the "To" input field by XPath and enter the police station coordinates
        to_input = driver.find_element(By.XPATH, '//*[@id="sb_ifc51"]/input')
        to_input.clear()
        to_input.send_keys(police_coords)
        to_input.send_keys("\n")  # Simulate pressing "Enter"

        # Wait for the results to load
        time.sleep(5)

        # Click on the driving option (car icon) if not already selected
        try:
            driving_icon = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="omnibox-directions"]/div/div[2]/div/div/div/div[2]/button'))
            )
            driving_icon.click()
            print("Clicked the 'Driving' icon successfully.")
        except TimeoutException:
            print("Could not click on the driving mode icon (Timeout).")
            return float('inf')  # Return a large number if driving mode can't be selected

        # Wait for a moment to let the page reload
        time.sleep(3)
        
        # Now, wait until the distance element is visible (timeout after 10 seconds)
        try:
            distance_element = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, 'div.ivN21e.tUEI8e.fontBodyMedium div'))
            )
            distance_text = distance_element.text  # Extract the distance (e.g., '650 m' or '2.4 km')
            print(f"Distance found: {distance_text}")
            return convert_to_meters(distance_text)
        except TimeoutException:
            print("Distance element not found after waiting for 10 seconds.")
            return float('inf')  # Return a large number in case of failure
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return float('inf')  # Return a very high value in case of an error to ensure it won't be selected as the minimum

# Initialize the WebDriver once
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Loop through the DataFrame and scrape the actual distance
for index, row in data.iterrows():
    atm_coords = row['ATM Coordinate (x, y)']
    police_coords = row['Police Station Coordinate (x, y)']
    
    # Ensure the coordinates are in string format (remove the parentheses)
    atm_coords = atm_coords.strip('()')
    police_coords = police_coords.strip('()')
    
    # Get the actual distance from Google Maps
    actual_distance = get_actual_distance(driver, atm_coords, police_coords)
    
    # Append the actual distance to the DataFrame
    data.at[index, 'Actual Distance (meters)'] = actual_distance
    
    # Progress tracking
    print(f"Completed {index + 1}/{len(data)} rows.")

# Close the browser
driver.quit()

# Save the updated DataFrame back to an Excel file
data.to_excel('updated_nearest_police_with_actual_distances.xlsx', index=False)

print("\nProcess completed successfully! All distances scraped and saved in 'updated_nearest_police_with_actual_distances.xlsx'.")


Fetching distance for ATM: 38.7728103, -9.1614562 to Police: 38.7763818, -9.1554179...
Clicked the 'Driving' icon successfully.
Distance found: 1.7 km
Completed 1/111 rows.
Fetching distance for ATM: 38.7872119, -9.180754 to Police: 38.78953, -9.1785879...
Clicked the 'Driving' icon successfully.
Distance found: 350 m
Completed 2/111 rows.
Fetching distance for ATM: 38.7339979, -9.145642 to Police: 38.7328938, -9.1592286...
Clicked the 'Driving' icon successfully.
Distance found: 2.5 km
Completed 3/111 rows.
Fetching distance for ATM: 38.7213309, -9.1487909 to Police: 38.7205021, -9.1558037...
Clicked the 'Driving' icon successfully.
Distance found: 1.3 km
Completed 4/111 rows.
Fetching distance for ATM: 38.7216269, -9.1502042 to Police: 38.7205021, -9.1558037...
Clicked the 'Driving' icon successfully.
Distance found: 1.0 km
Completed 5/111 rows.
Fetching distance for ATM: 38.7330105, -9.1450447 to Police: 38.7328938, -9.1592286...
Clicked the 'Driving' icon successfully.
Distance fou

In [4]:
data.to_excel('Actualpolicedistance.xlsx')

In [5]:
data

,Unnamed: 0,ATM Address,Bank,"ATM Coordinate (x, y)",Nearest Police Station,"Police Station Coordinate (x, y)",Distance (meters),Actual Distance (meters)
0,0,Alameda das Linhas de Torres,Banco BPI,"(38.7728103, -9.1614562)",PSP - 41st Squadron (Lumiar),"(38.7763818, -9.1554179)",657.06,"1,700.00"
1,1,"Av. Dom Dinis 68, 2675-328",Millennium bcp,"(38.7872119, -9.180754)",PSP - 71st Precinct (Odivelas),"(38.78953, -9.1785879)",318.89,350.00
2,2,Praça Duque de Saldanha 31 - B,"Balcão Santander Saldanha, Lisboa","(38.7339979, -9.145642)",PSP - 21th Squadron (Campolide),"(38.7328938, -9.1592286)","1,184.87","2,500.00"
3,3,R. Barata Salgueiro 30 5,Citibank International Plc-branch in Portugal,"(38.7213309, -9.1487909)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",615.33,"1,300.00"
4,4,R. Castilho 26 Piso 8,Banco Best - Centro de Investimento de Lisboa,"(38.7216269, -9.1502042)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",501.62,"1,000.00"
...,...,...,...,...,...,...,...,...
106,106,R. dos Soeiros 339 D,Millennium bcp,"(38.7560318, -9.177233)",PSP - Posto Policial do Centro Comercial Colombo,"(38.7557185, -9.1884517)",973.42,"1,300.00"
107,107,Av. Eng. Duarte Pacheco 21B,Balcão Santander Work Café Amoreiras,"(38.724257, -9.1591484)",PSP - 22ª Esquadra (Rato),"(38.7205021, -9.1558037)",508.45,"1,100.00"
108,108,Praça de Londres 3-B,novobanco Praça de Londres,"(38.7417434, -9.137523)",PSP - 12ª Esquadra (Olaias),"(38.7393457, -9.1259649)","1,037.29","2,200.00"
109,109,Alameda Manuel Ricardo Espírito Santo 2,EuroBic ABANCA - Benfica,"(38.7476, -9.18914)",PSP - 3rd Division of Lisbon,"(38.7457904, -9.1991176)",888.38,"1,700.00"
